In [ ]:
import shap
import matplotlib.pyplot as plt

def explain_model_with_shap(model, X_sample, feature_cols):
    """
    Generate SHAP explanations for LightGBM model
    """
    
    # Create explainer
    explainer = shap.TreeExplainer(model)
    
    # Calculate SHAP values (use sample if data is large)
    if len(X_sample) > 1000:
        X_sample = X_sample.sample(1000, random_state=42)
    
    shap_values = explainer.shap_values(X_sample)
    
    # Summary plot
    plt.figure(figsize=(12, 8))
    shap.summary_plot(shap_values, X_sample, feature_names=feature_cols, show=False)
    plt.title('SHAP Feature Importance')
    plt.tight_layout()
    plt.savefig('results/shap_summary.png', dpi=300, bbox_inches='tight')
    plt.close()
    
    # Bar plot (mean absolute SHAP)
    plt.figure(figsize=(10, 8))
    shap.summary_plot(shap_values, X_sample, feature_names=feature_cols, plot_type='bar', show=False)
    plt.title('Mean |SHAP| Values')
    plt.tight_layout()
    plt.savefig('results/shap_bar.png', dpi=300, bbox_inches='tight')
    plt.close()
    
    # Dependence plots for top features
    top_features = np.argsort(np.abs(shap_values).mean(axis=0))[-6:]
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()
    
    for idx, feature_idx in enumerate(top_features):
        feature_name = feature_cols[feature_idx]
        shap.dependence_plot(
            feature_idx, 
            shap_values, 
            X_sample,
            feature_names=feature_cols,
            ax=axes[idx],
            show=False
        )
        axes[idx].set_title(f'SHAP Dependence: {feature_name}')
    
    plt.tight_layout()
    plt.savefig('results/shap_dependence.png', dpi=300)
    plt.close()
    
    # Get feature importance ranking
    shap_importance = pd.DataFrame({
        'feature': feature_cols,
        'mean_abs_shap': np.abs(shap_values).mean(axis=0)
    }).sort_values('mean_abs_shap', ascending=False)
    
    print("\n=== TOP 20 FEATURES BY SHAP ===\n")
    print(shap_importance.head(20).to_string(index=False))
    
    shap_importance.to_csv('results/shap_feature_importance.csv', index=False)
    
    return shap_values, shap_importance

# Generate SHAP explanations
# Use first LightGBM model and validation data from first split
X_explain = monthly_data_prepared.iloc[splits[0]['val_idx']][feature_cols]

shap_values, shap_importance = explain_model_with_shap(
    lgb_models[0],
    X_explain,
    feature_cols
)